# Regime Adaptive Portfolio — Exploration

This notebook walks through the three core components of the project:
1. HMM regime detection — what the model learned and when it detected each regime
2. Optimizer comparison — how each optimizer allocates across assets
3. Backtest results — equity curve, drawdown, and performance metrics


In [ ]:
import sys
sys.path.append("..")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

plt.rcParams["figure.figsize"] = (14, 5)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3

REGIME_COLORS = {"Bull": "#2ecc71", "Bear": "#e74c3c", "Sideways": "#f39c12"}


## 1. Regime Detection

The HMM labels each trading day as Bull, Bear, or Sideways based on three features:
- **SPY log return** — daily return signal
- **Rolling volatility** — 21-day annualized volatility
- **Rolling correlation** — 63-day mean pairwise correlation across assets

In [ ]:
# Load data
regimes = pd.read_parquet("../data/regimes.parquet")
features = pd.read_parquet("../data/processed/features.parquet")
returns = pd.read_parquet("../data/processed/returns.parquet")

print(f"Date range: {regimes.index.min().date()} to {regimes.index.max().date()}")
print(f"Regime counts:\n{regimes['regime'].value_counts()}")

### 1.1 Regime Timeline

How the HMM classified each day over the full 20-year period.

In [ ]:
fig, ax = plt.subplots(figsize=(14, 3))

for regime, color in REGIME_COLORS.items():
    mask = regimes["regime"] == regime
    ax.fill_between(regimes.index, 0, 1,
                    where=mask,
                    color=color, alpha=0.6,
                    label=regime)

ax.set_yticks([])
ax.set_title("Market Regime Classification (Walk-Forward HMM)")
ax.legend(loc="upper left")
plt.tight_layout()
plt.show()

### 1.2 Feature Distributions by Regime

What each regime looks like in terms of the three input features.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

features_aligned = features.loc[regimes.index]
features_aligned["regime"] = regimes["regime"]

feature_cols = {
    "spy_return": "SPY Log Return",
    "spy_vol": "Rolling Volatility",
    "mean_corr": "Mean Correlation"
}

for ax, (col, label) in zip(axes, feature_cols.items()):
    for regime, color in REGIME_COLORS.items():
        subset = features_aligned[features_aligned["regime"] == regime][col]
        ax.hist(subset, bins=50, alpha=0.5, color=color, label=regime, density=True)
    ax.set_title(label)
    ax.legend()

plt.suptitle("Feature Distributions by Regime", y=1.02)
plt.tight_layout()
plt.show()

## 2. Optimizer Comparison

Each regime triggers a different optimizer. Here we compare the resulting asset allocations.

In [ ]:
from optimization.mean_var import max_sharpe
from optimization.risk_parity import risk_parity
from optimization.min_variance import min_variance

assets = returns.columns.tolist()

w_bull     = max_sharpe(returns)
w_bear     = risk_parity(returns)
w_sideways = min_variance(returns)

labels = ["Bull\n(Mean-Variance)", "Bear\n(Risk Parity)", "Sideways\n(Min Variance)"]
weights = [w_bull, w_bear, w_sideways]
colors  = [REGIME_COLORS["Bull"], REGIME_COLORS["Bear"], REGIME_COLORS["Sideways"]]

x = np.arange(len(assets))
width = 0.25

fig, ax = plt.subplots(figsize=(14, 5))
for i, (w, label, color) in enumerate(zip(weights, labels, colors)):
    ax.bar(x + i * width, w, width, label=label, color=color, alpha=0.8)

ax.set_xticks(x + width)
ax.set_xticklabels(assets)
ax.set_ylabel("Weight")
ax.set_title("Asset Allocation by Regime")
ax.legend()
plt.tight_layout()
plt.show()

## 3. Backtest Results

Portfolio performance versus SPY buy-and-hold over the full period.

In [ ]:
backtest = pd.read_parquet("../data/backtest_results.parquet")

common = backtest.index.intersection(returns.index)
spy_returns = returns.loc[common, "SPY"]
spy_equity = (1 + spy_returns).cumprod()

fig, axes = plt.subplots(2, 1, figsize=(14, 10))

# Equity curves
axes[0].plot(backtest["equity"], label="Regime Portfolio", color="#3498db", linewidth=1.5)
axes[0].plot(spy_equity, label="SPY Buy & Hold", color="#e74c3c", linewidth=1.5, alpha=0.7)
axes[0].set_title("Equity Curve")
axes[0].set_ylabel("Growth of $1")
axes[0].legend()

# Drawdown
port_dd = (backtest["equity"] - backtest["equity"].cummax()) / backtest["equity"].cummax()
spy_dd  = (spy_equity - spy_equity.cummax()) / spy_equity.cummax()

axes[1].fill_between(port_dd.index, port_dd, 0, alpha=0.4, color="#3498db", label="Portfolio")
axes[1].fill_between(spy_dd.index, spy_dd, 0, alpha=0.4, color="#e74c3c", label="SPY")
axes[1].set_title("Drawdown")
axes[1].set_ylabel("Drawdown")
axes[1].legend()

plt.tight_layout()
plt.show()

### 3.1 Performance Metrics

In [ ]:
from backtest.metrics import compute_all

port_metrics = compute_all(backtest["portfolio_return"], backtest["equity"])
spy_metrics  = compute_all(spy_returns, spy_equity)

metrics_df = pd.DataFrame({
    "Portfolio": port_metrics,
    "SPY": spy_metrics
})

metrics_df.index = [
    "Annualized Return",
    "Annualized Volatility",
    "Sharpe Ratio",
    "Max Drawdown",
    "Calmar Ratio"
]

metrics_df